In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

pd.set_option("display.max_columns", None)

In [2]:
def load_data(df_path):
    df = pd.read_csv(df_path, low_memory=False)

    df.rename(
        columns={
            "Sales Date": "date",
            "Outlet SF ID": "customer_code",
            "Store Participant Code": "customer_name",
            "SKU SF ID": "sku_code",
            "SKU Name": "sku_name",
            "Brand Variant": "brand_variant",
            "Brand Family": "brand_name",
            "Category": "category",
            "Volume in Unit": "sales_amount",
            "Volume in Packs": "sales_quantity",
            "Ownership Type": "channel_name",
            "Latitude": "latitude",
            "Longitude": "longitude",
            "Territory Id": "route",
            "Brand": "brand",
            "SKU Clean": "sku_clean",
            "Month": "month",
        },
        inplace=True,
    )

    df["date"] = pd.to_datetime(df["date"])

    df_sellin = df[df["data_type"] == "sell_in"].copy()
    df_sellout = df[df["data_type"] == "sell_out"].copy()

    return df_sellin, df_sellout

In [3]:
df_combined_path = "../../../data/France/processed_data/combined_df.csv"
# df = pd.read_csv(df_path)
# df = pd.read_csv(df_combined_path, low_memory=False)
df_sellin, df_sellout = load_data(df_combined_path)

In [4]:
df_sellout.info()

<class 'pandas.core.frame.DataFrame'>
Index: 929460 entries, 122307 to 1051766
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   data_type       929460 non-null  object        
 1   source          929460 non-null  object        
 2   date            929460 non-null  datetime64[ns]
 3   customer_code   929460 non-null  object        
 4   customer_name   928877 non-null  float64       
 5   sku_code        929460 non-null  object        
 6   sku_name        790922 non-null  object        
 7   brand_variant   790922 non-null  object        
 8   brand_name      790922 non-null  object        
 9   category        760326 non-null  object        
 10  sales_amount    929460 non-null  int64         
 11  sales_quantity  929460 non-null  float64       
 12  channel_name    926998 non-null  object        
 13  latitude        928877 non-null  float64       
 14  longitude       928877 non-null  fl

In [5]:
def check_null_customers(df_sellout):
    total_sales_amount_so = df_sellout["sales_amount"].sum()
    custormer_code_null_count = df_sellout["customer_code"].isnull().sum()
    print(
        f"Number of null values in 'customer_code': {custormer_code_null_count}, which is {round(custormer_code_null_count/len(df_sellout)*100, 2)}% of the total records."
    )
    print(
        "The total sell for null customer code is: ",
        df_sellout[df_sellout["customer_code"].isnull()]["sales_amount"].sum(),
    )
    print(
        "Their sale is only: ",
        round(
            df_sellout[df_sellout["customer_code"].isnull()]["sales_amount"].sum()
            / total_sales_amount_so
            * 100,
            2,
        ),
        "%",
    )

In [6]:
def check_null_sku(df):
    total_sales_amount = df_sellout["sales_amount"].sum()
    sku_code_null_count = df[df["sku_code"] == "0"].shape[0]
    print(
        f"Number of null values in 'sku_code': {sku_code_null_count}, which is {round(sku_code_null_count/len(df)*100, 2)}% of the total records."
    )
    print(
        "The total sell for null sku code is: ",
        df[df["sku_code"] == "0"]["sales_amount"].sum(),
    )
    print(
        "Their sale is only: ",
        round(
            df[df["sku_code"] == "0"]["sales_amount"].sum() / total_sales_amount * 100,
            2,
        ),
        "%",
    )

In [8]:
check_null_customers(df_sellout)

Number of null values in 'customer_code': 0, which is 0.0% of the total records.
The total sell for null customer code is:  0
Their sale is only:  0.0 %


In [9]:
df_sellout.customer_code.value_counts()

customer_code
0011t000011b5TpAAI    3326
0011t000011b2hEAAQ    3119
0011t000011b2f6AAA    3016
0011t000011b52lAAA    2903
0011t000011b5IpAAI    2892
                      ... 
a0f1t000002XnmwAAC      26
a0f1t000002XpYJAA0      25
a0f1t000002XrJoAAK      25
a0f1t000002XoqnAAC      25
a0f1t000002XodNAAS       8
Name: count, Length: 612, dtype: int64

In [10]:
df_sellout.customer_name.value_counts()

customer_name
305032.0    3326
306009.0    3119
304080.0    3016
302153.0    2903
304081.0    2892
            ... 
301182.0     556
306122.0     554
303110.0     529
304114.0     320
307086.0     299
Name: count, Length: 603, dtype: int64

In [11]:
check_null_sku(df_sellout)

Number of null values in 'sku_code': 133073, which is 14.32% of the total records.
The total sell for null sku code is:  258718952
Their sale is only:  85.13 %


In [12]:
check_null_sku(df_sellin)

Number of null values in 'sku_code': 21804, which is 17.83% of the total records.
The total sell for null sku code is:  47652915
Their sale is only:  15.68 %


## Common in Both

### Customer Level Selling Difference

In [13]:
customers_si = set(df_sellin.customer_code.unique())
customers_so = set(df_sellout.customer_code.unique())

common_customers = customers_si.intersection(customers_so)

print(
    f"Total Customers SellOut: {len(customers_so)} and SellIn: {len(customers_si)} and Common: {len(common_customers)}"
)

Total Customers SellOut: 612 and SellIn: 650 and Common: 602


In [14]:
sellin_common = df_sellin[df_sellin["customer_code"].isin(common_customers)]
sellout_common = df_sellout[df_sellout["customer_code"].isin(common_customers)]

In [15]:
sellout_sell_by_customer = sellout_common.groupby("customer_code")[
    "sales_quantity"
].sum()
sellin_sell_by_customer = sellin_common.groupby("customer_code")["sales_quantity"].sum()

selling = pd.DataFrame(
    {
        "sellout": sellout_sell_by_customer,
        "sellin": sellin_sell_by_customer,
    }
)

In [16]:
selling["si_is_x_time_so"] = (selling["sellin"] - selling["sellout"]) / selling[
    "sellin"
]

In [17]:
selling.sort_values("si_is_x_time_so", ascending=True).head()

,sellout,sellin,si_is_x_time_so
customer_code,,,
0011t000011b1xXAAQ,13895.0,985.0,-13.106599
0011t000011b4vzAAA,49612.0,5780.0,-7.583391
0011t000011bCLNAA2,30251.0,4426.0,-5.834840
0011t000011b6A0AAI,14171.0,2224.0,-5.371853
0011t000011b8zmAAA,17513.0,3012.0,-4.814409


### SKU Level Selling Difference

In [19]:
sellin_sku_common = df_sellin.groupby(["customer_code", "sku_code"])[
    "sales_quantity"
].sum()
sellout_sku_common = df_sellout.groupby(["customer_code", "sku_code"])[
    "sales_quantity"
].sum()

In [20]:
selling_sku = pd.DataFrame(
    {
        "sellout": sellout_sku_common,
        "sellin": sellin_sku_common,
    }
)

In [21]:
selling_sku["si_is_x_time_so"] = (selling_sku["sellin"] - selling_sku["sellout"]) / (
    selling_sku["sellin"] + 2
)

In [131]:
selling_sku.sort_values("si_is_x_time_so", ascending=False).tail()

sellout  sellin  si_is_x_time_so
customer_code      sku_code                                            
a0f1t000002XqBDAA0 a0U3W000000RV8dUAG      9.0     NaN              NaN
                   a0U3W000002BYaWUAW     39.0     NaN              NaN
                   a0U3W000002BYaXUAW     86.0     NaN              NaN
a0f1t000002XqndAAC 0                     477.0     NaN              NaN
a0f1t000002XrJoAAK 0                     184.0     NaN              NaN

### Taking only FMC Category

#### SKU-Level Analysis

In [100]:
sellin_common_fmc = sellin_common[sellin_common["category"] == "FMC"].copy()
sellout_common_fmc = sellout_common[sellout_common["category"] == "FMC"].copy()

In [101]:
sellin_fmc_unique_fmc = set(sellin_common_fmc["sku_code"].unique())
sellout_fmc_unique_fmc = set(sellout_common_fmc["sku_code"].unique())

common_fmc = sellin_fmc_unique_fmc.intersection(sellout_fmc_unique_fmc)

print(
    f"Total FMC SellOut: {len(sellout_fmc_unique_fmc)} and SellIn: {len(sellin_fmc_unique_fmc)} and Common: {len(common_fmc)}"
)

sellin_common_fmc = sellin_common_fmc[sellin_common_fmc["sku_code"].isin(common_fmc)]
sellout_common_fmc = sellout_common_fmc[sellout_common_fmc["sku_code"].isin(common_fmc)]

Total FMC SellOut: 43 and SellIn: 40 and Common: 40


In [104]:
sellin = sellin_common_fmc.groupby("sku_code")["sales_quantity"].sum()
sellout = sellout_common_fmc.groupby("sku_code")["sales_quantity"].sum()


selling = pd.DataFrame(
    {
        "sellout": sellout,
        "sellin": sellin,
    }
)
selling["si_is_x_time_so"] = (selling["sellin"] - selling["sellout"]) / selling[
    "sellin"
]

In [111]:
selling.sort_values("si_is_x_time_so", ascending=True).head()

,sellout,sellin,si_is_x_time_so
sku_code,,,
a0U1t000002PXpqEAG,4623.0,4040.0,-0.144307
a0U1t000002PXnGEAW,6341.0,5850.0,-0.083932
a0U1t000002PXpKEAW,14598.0,13650.0,-0.069451
a0U3W000000TPMlUAO,23053.0,21570.0,-0.068753
a0U3W000000TPMmUAO,4507.0,4290.0,-0.050583


In [112]:
print(
    f"Total Sellin FMC Sales: {sellin.sum()}, Sellout FMC Sales: {sellout.sum()}, PCT Diff: {round((sellin.sum() - sellout.sum()) / sellin.sum() * 100, 2)}%"
)

Total Sellin FMC Sales: 2110650.0, Sellout FMC Sales: 2119864.0, PCT Diff: -0.44%


#### Customer Level Analysis

In [124]:
sellin = sellin_common_fmc.groupby("customer_code")["sales_quantity"].sum()
sellout = sellout_common_fmc.groupby("customer_code")["sales_quantity"].sum()

selling = pd.DataFrame(
    {
        "sellout": sellout,
        "sellin": sellin,
    }
)
selling["si_is_x_time_so"] = (selling["sellin"] - selling["sellout"]) / selling[
    "sellin"
]

In [22]:
# selling.sort_values("si_is_x_time_so", ascending=True).head(50)

In [135]:
df_sellout.sku_code.nunique(), df_sellin.sku_code.nunique()

(305, 202)

#### Check for SKU IN FMC

In [23]:
df_sellin, df_sellout = load_data(df_combined_path)

In [24]:
df_sellin_fmc = df_sellin[df_sellin["category"] == "FMC"].copy()
df_sellout_fmc = df_sellout[df_sellout["category"] == "FMC"].copy()

In [27]:
total_sellin_fmc_sku = set(df_sellin_fmc["sku_code"].unique())
total_sellout_fmc_sku = set(df_sellout_fmc["sku_code"].unique())
common_fmc_sku = total_sellin_fmc_sku.intersection(total_sellout_fmc_sku)


print(
    f"Total FMC SellOut: {len(total_sellout_fmc_sku)} and SellIn: {len(total_sellin_fmc_sku)} and Common: {len(common_fmc_sku)}"
)

Total FMC SellOut: 43 and SellIn: 40 and Common: 40


In [30]:
df_sellin_fmc.columns

Index(['data_type', 'source', 'date', 'customer_code', 'customer_name',
       'sku_code', 'sku_name', 'brand_variant', 'brand_name', 'category',
       'sales_amount', 'sales_quantity', 'channel_name', 'latitude',
       'longitude', 'route', 'brand', 'month'],
      dtype='object')

In [54]:
import pandas as pd
import numpy as np


def calculate(
    common_fmc_sku: list,
    df_sellin_fmc: pd.DataFrame,
    df_sellout_fmc: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build a summary DataFrame for each (sku_code, customer_code) pair.
    For each SKU, loops through all customers that have that SKU in sellin.
    """

    records = []

    for sku in common_fmc_sku:

        # --- filter to this SKU ---------------------------------------
        si_sku = df_sellin_fmc[df_sellin_fmc["sku_code"] == sku]
        so_sku = df_sellout_fmc[df_sellout_fmc["sku_code"] == sku]

        if si_sku.empty:
            continue

        # --- loop through each customer for this SKU ------------------
        for customer in so_sku["customer_code"].unique():

            si_raw = si_sku[si_sku["customer_code"] == customer]
            so_raw = so_sku[so_sku["customer_code"] == customer]

            # --- aggregate to daily grain -----------------------------
            si_grp = si_raw.groupby("date", as_index=False)["sales_quantity"].sum()
            so_grp = so_raw.groupby("date", as_index=False)["sales_quantity"].sum()

            si_qty = si_grp["sales_quantity"]
            si_date = si_grp["date"]

            # --- date ranges -----------------------------------------
            si_startdate = si_date.min()
            si_enddate = si_date.max()
            so_startdate = so_grp["date"].min() if not so_grp.empty else None
            so_enddate = so_grp["date"].max() if not so_grp.empty else None

            # --- dips (sales_quantity > 0) ----------------------------
            si_dip_mask = si_qty > 0
            si_totaldips = int(si_dip_mask.sum())

            # dips in the last month of the sellin window
            si_dipinlastmonth = int(
                (
                    si_dip_mask
                    & (si_date.dt.month == si_enddate.month)
                    & (si_date.dt.year == si_enddate.year)
                ).sum()
            )

            # --- day counts ------------------------------------------
            total_si_days = si_date.nunique()
            total_so_days = so_grp["date"].nunique() if not so_grp.empty else 0

            # --- sellin quantity stats --------------------------------
            si_total_qty = si_qty.sum()
            si_avg_qty_per_dipday = (
                float(si_qty[si_dip_mask].mean()) if si_totaldips > 0 else 0.0
            )
            si_max_qty = si_qty.max()

            # --- sellout quantity stats -------------------------------
            if not so_grp.empty:
                so_qty = so_grp["sales_quantity"]
                so_total_qty = so_qty.sum()
                so_avg_qty = float(so_qty.mean())
                so_max_qty = so_qty.max()
            else:
                so_total_qty = so_avg_qty = so_max_qty = 0.0

            si_so_ratio = (
                round(si_total_qty / so_total_qty, 4) if so_total_qty else np.nan
            )

            records.append(
                {
                    "sku_code": sku,
                    "customer_code": customer,
                    "si_startdate": si_startdate,
                    "si_enddate": si_enddate,
                    "so_startdate": so_startdate,
                    "so_enddate": so_enddate,
                    "si_totaldips": si_totaldips,
                    "si_dipinlastmonth": si_dipinlastmonth,
                    "total_si_days": total_si_days,
                    "total_so_days": total_so_days,
                    "si_total_quantity": si_total_qty,
                    "si_avg_qty_per_dipday": round(si_avg_qty_per_dipday, 4),
                    "si_max_quantity": si_max_qty,
                    "so_total_quantity": so_total_qty,
                    "so_avg_qty_per_day": round(so_avg_qty, 4),
                    "so_max_quantity": so_max_qty,
                    "si_so_quantity_ratio": si_so_ratio,
                }
            )

    return pd.DataFrame(records).reset_index(drop=True)

In [55]:
result = calculate(common_fmc_sku, df_sellin_fmc, df_sellout_fmc)

In [60]:
# result.sort_values("so_total_quantity", ascending=False).head()
result.head()

,sku_code,customer_code,si_startdate,si_enddate,so_startdate,so_enddate,si_totaldips,si_dipinlastmonth,total_si_days,total_so_days,si_total_quantity,si_avg_qty_per_dipday,si_max_quantity,so_total_quantity,so_avg_qty_per_day,so_max_quantity,si_so_quantity_ratio
0,a0U1t000002PXnGEAW,0011t000011bBYUAA2,2026-01-16,2026-02-13,2026-01-07,2026-03-30,2,1,2,11,20.0,10.0,10.0,21.0,1.9091,2.0,0.9524
1,a0U1t000002PXnGEAW,0011t000011b36MAAQ,2026-01-21,2026-03-18,2026-01-02,2026-03-31,5,2,5,67,110.0,22.0,30.0,141.0,2.1045,7.0,0.7801
2,a0U1t000002PXnGEAW,0011t000011b60JAAQ,2026-01-16,2026-03-27,2026-01-03,2026-03-25,4,1,4,22,90.0,22.5,40.0,80.0,3.6364,12.0,1.1250
3,a0U1t000002PXnGEAW,0011t000011b8BYAAY,2026-02-13,2026-02-13,2026-01-07,2026-03-25,1,1,1,14,10.0,10.0,10.0,14.0,1.0000,1.0,0.7143
4,a0U1t000002PXnGEAW,0011t000011b1vGAAQ,2026-01-15,2026-01-15,2026-02-02,2026-03-31,1,1,1,4,10.0,10.0,10.0,4.0,1.0000,1.0,2.5000


In [58]:
result.shape

(17850, 17)

##### Extra Analysis (ALEEM)

In [90]:
result = []

shops = df_sellout_fmc["customer_code"].unique()

total_days = df_sellout_fmc["date"].nunique()
total_company_sales = df_sellout_fmc["sales_quantity"].sum()

for shop in shops:

    shop_df = df_sellout_fmc[df_sellout_fmc["customer_code"] == shop].copy()
    total_sale = shop_df["sales_quantity"].sum()
    total_amount = shop_df["sales_amount"].sum()

    unique_skus = shop_df["sku_code"].nunique()
    avg_sales_per_sku = total_sale / unique_skus if unique_skus > 0 else 0
    active_days = shop_df["date"].nunique()
    purchase_frequency = active_days / total_days if total_days != 0 else 0

    daily_sales = shop_df.groupby("date")["sales_quantity"].sum()

    mean_sales = daily_sales.mean()
    std_sales = daily_sales.std()

    cv = std_sales / mean_sales if mean_sales != 0 else 0

    sku_sales = (
        shop_df.groupby("sku_code")["sales_quantity"].sum().sort_values(ascending=False)
    )

    top_sku = sku_sales.index[0]
    top_sku_sales = sku_sales.iloc[0]
    top_sku_pct = top_sku_sales / total_sale if total_sale != 0 else 0
    top_sku_df = shop_df[shop_df["sku_code"] == top_sku]

    top_sku_daily = top_sku_df.groupby("date")["sales_quantity"].sum()

    top_sku_mean = top_sku_daily.mean()
    top_sku_std = top_sku_daily.std()

    top_sku_cv = top_sku_std / top_sku_mean if top_sku_mean != 0 else 0

    top_sku_active_days = top_sku_df["date"].nunique()

    top_sku_purchase_frequency = (
        top_sku_active_days / active_days if active_days != 0 else 0
    )

    sku_share = sku_sales / total_sale if total_sale != 0 else 0

    diversification = -(sku_share * np.log(sku_share + 1e-9)).sum()

    normalized_diversification = (
        diversification / np.log(unique_skus) if unique_skus > 1 else 0
    )

    contribution = total_sale / total_company_sales if total_company_sales != 0 else 0

    high_dependency_flag = top_sku_pct > 0.6
    low_activity_flag = purchase_frequency < 0.1
    unstable_shop_flag = cv > 1
    unstable_top_sku_flag = top_sku_cv > 1

    result.append(
        {
            "customer_code": shop,
            "total_sale": total_sale,
            "total_amount": total_amount,
            "unique_skus": unique_skus,
            "avg_sales_per_sku": avg_sales_per_sku,
            "purchase_frequency": purchase_frequency,
            "cv": cv,
            "top_sku": top_sku,
            "top_sku_pct": top_sku_pct,
            "top_sku_cv": top_sku_cv,
            "top_sku_purchase_frequency": top_sku_purchase_frequency,
            # "diversification": diversification,
            # "normalized_diversification": normalized_diversification,
            "contribution": contribution,
            "high_dependency_flag": high_dependency_flag,
            "low_activity_flag": low_activity_flag,
            "unstable_shop_flag": unstable_shop_flag,
            "unstable_top_sku_flag": unstable_top_sku_flag,
        }
    )

/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [91]:
df_shop_analysis = pd.DataFrame(result)
df_shop_analysis.head()

,customer_code,total_sale,total_amount,unique_skus,avg_sales_per_sku,purchase_frequency,cv,top_sku,top_sku_pct,top_sku_cv,top_sku_purchase_frequency,contribution,high_dependency_flag,low_activity_flag,unstable_shop_flag,unstable_top_sku_flag
0,0011t000011b5rcAAA,753.0,15105,17,44.294118,0.903226,0.417915,a0U1t000002PXlqEAG,0.199203,0.572831,0.714286,0.000355,False,False,False,False
1,0011t000011b288AAA,2946.0,59860,30,98.200000,0.946237,0.331857,a0U1t000002PXjoEAG,0.146979,0.826398,0.943182,0.001388,False,False,False,False
2,0011t000011bCLOAA2,3904.0,78465,32,122.000000,0.924731,0.266972,a0U3W000002BYaXUAW,0.173668,0.458997,0.988372,0.001840,False,False,False,False
3,0011t000011b285AAA,4204.0,85565,39,107.794872,0.763441,0.240201,a0U1t000002PXlqEAG,0.159134,0.575633,1.000000,0.001981,False,False,False,False
4,0011t000011b1niAAA,972.0,19640,25,38.880000,0.946237,0.437952,a0U3W000002BYaXUAW,0.300412,0.548384,0.875000,0.000458,False,False,False,False


In [92]:
new_df_path = "/home/asad/Downloads/predictive.csv"
df_new = pd.read_csv(new_df_path)

In [94]:
df_new = df_new[df_new["status"] == "Processed"]

df_new.head()

,sec,channel,route,customer,sku,sku_sales_contrib,status,reason,second_last_sellin_date,second_last_sellin_sales,days_difference,forecasted_sales,actual_sellout_sales,val_difference
2,A,COCO,a0D1t000005iL9pEAE,306055.0,1201 - LUCKY STRIKE BLEU EN 20,0.046377,Processed,NaN,2026-03-19,600.0,7.0,904.381673,700.0,-204.381673
3,A,COCO,a0D1t000005iL9pEAE,306055.0,125 - LUCKY STRIKE RED 20,0.127914,Processed,NaN,2026-03-12,5000.0,7.0,2482.829422,1840.0,-642.829422
5,A,COCO,a0D1t000005iL9pEAE,306055.0,2520 - DUNHILL INTERNATIONAL ROUGE EN 20,0.031002,Processed,NaN,2026-03-12,200.0,14.0,1349.267596,900.0,-449.267596
6,A,COCO,a0D1t000005iL9pEAE,306055.0,2590 - DUNHILL ROUGE EN 20,0.039950,Processed,NaN,2026-03-19,1000.0,7.0,668.871826,660.0,-8.871826
13,A,COCO,a0D1t000005iL9pEAE,306055.0,47762 - PALL MALL EN 30G,0.020605,Processed,NaN,2026-03-19,300.0,7.0,448.491651,360.0,-88.491651


In [103]:
df_filtered = df_new[
    (df_new["actual_sellout_sales"] >= df_new["actual_sellout_sales"].quantile(0.70))
    & (df_new["actual_sellout_sales"] <= df_new["actual_sellout_sales"].quantile(1.00))
].sort_values("actual_sellout_sales")

In [107]:
df_filtered.shape, df_new.shape

((1459, 14), (4854, 14))

In [ ]:
df_filtered.iloc[1300:1310]

,sec,channel,route,customer,sku,sku_sales_contrib,status,reason,second_last_sellin_date,second_last_sellin_sales,days_difference,forecasted_sales,actual_sellout_sales,val_difference
9300,B,COCO,a0D1t000005iLAGEA2,304173.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.211824,Processed,NaN,2026-03-16,2000.0,10.0,3299.179034,2760.0,-539.179034
3192,A,COCO,a0D1t000005iLAnEAM,300007.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.270238,Processed,NaN,2026-03-12,2400.0,14.0,3191.851824,2760.0,-431.851824
3919,A,COCO,a0D1t000005iLBVEA2,302153.0,6414 - VOGUE ORIGINALE PASTEL EN 20,0.073419,Processed,NaN,2026-03-13,5000.0,14.0,3005.847616,2800.0,-205.847616
17616,C,COCO,a0D1t000005iLAAEA2,308124.0,125 - LUCKY STRIKE RED 20,0.175176,Processed,NaN,2026-03-13,3600.0,14.0,1644.613705,2800.0,1155.386295
45034,E,COCO,a0D1t000005iLBVEA2,304092.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.155207,Processed,NaN,2026-03-13,2000.0,14.0,2082.030018,2800.0,717.969982
37840,D,COCO,a0D1t000005iLBVEA2,304143.0,85548 - VOGUE Lâ€™ORIGINALE VERTE ICE 20s,0.188348,Processed,NaN,2026-03-12,3200.0,14.0,2369.058728,2800.0,430.941272
22177,C,COCO,a0D1t000005iLAnEAM,303109.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.189673,Processed,NaN,2026-03-19,1600.0,7.0,2437.294656,2800.0,362.705344
5271,A,Independent,a0D1t000005iLBVEA2,302015.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.226941,Processed,NaN,2026-03-11,2800.0,14.0,2929.805857,2800.0,-129.805857
4970,A,Independent,a0D1t000005iLAAEA2,308151.0,125 - LUCKY STRIKE RED 20,0.187174,Processed,NaN,2026-03-13,2000.0,14.0,2353.579872,2800.0,446.420128
26230,C,Independent,a0D1t000005iL9pEAE,306101.0,6414 - VOGUE ORIGINALE PASTEL EN 20,0.164773,Processed,NaN,2026-03-11,1200.0,14.0,1770.671870,2820.0,1049.328130


In [117]:
indexes = [36243, 17964, 35735, 25302, 10349, 19421, 18322, 17639, 3192, 22177]

df_selected = df_filtered.loc[indexes]

In [118]:
df_selected

,sec,channel,route,customer,sku,sku_sales_contrib,status,reason,second_last_sellin_date,second_last_sellin_sales,days_difference,forecasted_sales,actual_sellout_sales,val_difference
36243,D,COCO,a0D1t000005iLBFEA2,304029.0,125 - LUCKY STRIKE RED 20,0.125343,Processed,NaN,2026-03-13,3000.0,14.0,3465.746339,2380.0,-1085.746339
17964,C,COCO,a0D1t000005iLAGEA2,300058.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.196113,Processed,NaN,2026-03-19,2000.0,7.0,2612.248613,2400.0,-212.248613
35735,D,COCO,a0D1t000005iLBFEA2,304002.0,1201 - LUCKY STRIKE BLEU EN 20,0.122328,Processed,NaN,2026-03-13,1400.0,14.0,1327.261954,2380.0,1052.738046
25302,C,COCO,a0D1t000005iLBVEA2,309199.0,85548 - VOGUE Lâ€™ORIGINALE VERTE ICE 20s,0.116006,Processed,NaN,2026-03-13,2000.0,14.0,2078.226158,1840.0,-238.226158
10349,B,COCO,a0D1t000005iLAnEAM,301100.0,6414 - VOGUE ORIGINALE PASTEL EN 20,0.128843,Processed,NaN,2026-03-04,1000.0,14.0,1215.318291,1840.0,624.681709
19421,C,COCO,a0D1t000005iLAGEA2,306171.0,125 - LUCKY STRIKE RED 20,0.081408,Processed,NaN,2026-03-12,1600.0,14.0,1769.729954,1880.0,110.270046
18322,C,COCO,a0D1t000005iLAGEA2,300127.0,85548 - VOGUE Lâ€™ORIGINALE VERTE ICE 20s,0.187780,Processed,NaN,2026-03-19,600.0,7.0,2226.162418,2080.0,-146.162418
17639,C,COCO,a0D1t000005iLAAEA2,308124.0,85548 - VOGUE Lâ€™ORIGINALE VERTE ICE 20s,0.109311,Processed,NaN,2026-03-13,1600.0,14.0,1943.026194,2080.0,136.973806
3192,A,COCO,a0D1t000005iLAnEAM,300007.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.270238,Processed,NaN,2026-03-12,2400.0,14.0,3191.851824,2760.0,-431.851824
22177,C,COCO,a0D1t000005iLAnEAM,303109.0,3721 - VOGUE ORIGINALE BLEUE EN 20,0.189673,Processed,NaN,2026-03-19,1600.0,7.0,2437.294656,2800.0,362.705344


In [120]:
customers_selected = df_selected["customer"].unique()